## Domain Adaptation

In [1]:
import os
from pathlib import Path
import yaml
import time
import scipy
import json
import sys
sys.path.append("/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/Domain-Adaptation")
from embedding import EmbeddingManager
from utils import get_domain_args

import numpy as np
from PIL import Image
from typing import Dict, List, Callable, Any, Optional, Tuple
import numpy.typing as npt

from transformers import CLIPModel
from blora import BlockLoraConfig, get_blocklora_model, BlockLoraModel
from peft import LoraConfig, get_peft_model, PeftModel

# adjust these paths as needed
output_dir = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz"

# Trained LoRA path
lora_library_path = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/lora-db"
blora_library_path = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db"

import torch, gc
torch.cuda.set_device(9)
torch.cuda.empty_cache()
gc.collect()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_CLASSES = 20
IMG_SIZE = (224, 224)

os.path.exists(output_dir)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


True

In [2]:
BlockLoraConfig().Cheer

'ฺBeta'

In [3]:
# Dataset Path
acdc_dir = "/home/chenyinjia/data/dataset/acdc"
ade20k_dir = "/home/chenyinjia/data/dataset/ADE20ID"
cs_dir = "/home/chenyinjia/data/dataset/cityscapes"
muse_dir = "/home/chenyinjia/data/dataset/muses"
bdd_dir = "/home/chenyinjia/data/dataset/bdd100k"
mv_dir = "/home/chenyinjia/data/dataset/mapillary_vistas"


# base model checkpoint
clip32_dir = "/home/chenyinjia/clip-vit-large-patch14"
MODEL_PATH = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/checkpoints3/L14m7.pth"


# TTA config
semla_config_dir = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/Domain-Adaptation/config/semla_config.yaml"
source_domains_dir = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/Domain-Adaptation/config/source_domains.yaml"
target_domains_dir = "/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/Domain-Adaptation/config/target_domains.yaml"

## Dataset

In [4]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

# ============================================================
# 🔧 ADE20K official decoder
# ============================================================
def ade20k_rgb_to_class(mask_rgb: np.ndarray):
    """Decode ADE20K RGB mask to class IDs (official MIT method)."""
    mask_id = (
        mask_rgb[:, :, 0].astype(np.int32)
        + 256 * mask_rgb[:, :, 1].astype(np.int32)
        + 256 * 256 * mask_rgb[:, :, 2].astype(np.int32)
    )
    class_id = mask_id % 256
    return class_id.astype(np.int64)

# --- Cityscapes label set (index reference) ---
cityscapes_labels = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light", "traffic sign",
    "vegetation", "terrain", "sky", "person", "rider", "car", "truck", "bus", 
    "train", "motorcycle", "bicycle", "unknown"
]

# --- ADE20K → Cityscapes Unified Mapping ---
ADE20K_TO_CITYSCAPES = {
    0: 255,   # background → unknown
    1: 255,    # wall
    2: 2,    # building
    3: 10,   # sky
    4: 255,    # floor → unknown
    5: 8,    # tree → vegetation
    6: 255,   # ceiling → unknown
    7: 0,    # road
    8: 255,   # bed
    9: 255,   # windowpane
    10: 8,   # grass → vegetation
    11: 255,  # cabinet → unknown
    12: 1,   # sidewalk
    13: 11,  # person
    14: 9,   # earth → terrain
    15: 255,   # door → unknown
    16: 255,  # table
    17: 255,   # mountain → terrain
    18: 8,   # plant → vegetation
    19: 255,  # curtain
    20: 255,  # chair
    21: 13,  # car
    22: 255,   # water → terrain
    23: 255,  # painting
    24: 255,  # sofa
    25: 255,  # shelf
    26: 2,   # house → building
    27: 255,   # sea → terrain
    28: 255,  # mirror
    29: 255,   # rug → terrain
    30: 9,   # field → terrain
    31: 255,  # armchair
    32: 255,  # seat
    33: 4,   # fence
    34: 255,  # desk
    35: 255,   # rock → terrain
    36: 255,  # wardrobe
    37: 255,  # lamp
    38: 255,  # bathtub
    39: 4,   # railing → fence
    40: 255,  # pillow
    41: 255,  # base
    42: 255,  # box
    43: 5,   # column → pole
    44: 7,   # signboard → traffic sign
    45: 255,  # chest of drawers
    46: 255,  # counter
    47: 9,   # sand → terrain
    48: 255,  # sink
    49: 2,   # skyscraper → building
    50: 255,  # fireplace
    51: 255,  # refrigerator
    52: 255,  # grandstand
    53: 0,   # path → road
    54: 255,  # stairs
    55: 0,   # runway → road
    56: 255,  # case
    57: 255,  # pool table
    58: 255,  # pillow
    59: 255,   # screen door → terrain
    60: 255, # stairway
    61: 255,   # river → terrain
    62: 9,   # bridge → building
    63: 255,  # bookcase 
    64: 255,  # blind
    65: 255,  # coffee table
    66: 255,  # toilet
    67: 8,    # flower → vegetation
    68: 255,  # book
    69: 9,    # hill → terrain
    70: 255,  # bench
    71: 255,  # countertop
    72: 255,  # stove
    73: 8,    # palm → vegetation
    74: 255,  # kitchen island
    75: 255,  # computer
    76: 255,  # swivel chair
    77: 255,  # boat → bicycle (closest moving object type)
    78: 255,  # bar
    79: 255,  # arcade machine
    80: 2,   # hovel → building
    81: 15,  # bus
    82: 255,  # towel
    83: 255,  # light
    84: 14,  # truck
    85: 2,   # tower → building
    86: 255,  # chandelier
    87: 5,   # awning → pole
    88: 5,   # streetlight → pole
    89: 255,  # booth
    90: 255,  # television receiver
    91: 255,  # airplane → motorcycle (generic vehicle)
    92: 0,   # dirt track → road
    93: 255,  # apparel
    94: 5,   # pole
    95: 9,   # land → terrain
    96: 4,   # bannister → fence
    97: 255,  # escalator
    98: 255,  # ottoman
    99: 255,  # bottle
    100: 255, # buffet
    101: 255, # poster
    102: 255, # stage
    103: 15, # van → bus
    104: 255, # ship → bicycle (vehicle)
    105: 255,  # fountain → terrain
    106: 255, # conveyer belt
    107: 255, # canopy
    108: 255, # washer
    109: 255, # plaything
    110: 255,  # swimming pool → terrain
    111: 255, # stool
    112: 255, # barrel
    113: 255, # basket
    114: 255,  # waterfall → terrain
    115: 255, # tent
    116: 255, # bag
    117: 17, # minibike → motorcycle
    118: 255, # cradle
    119: 255, # oven
    120: 255, # ball
    121: 255, # food
    122: 255, # step
    123: 255, # tank (ambiguous)
    124: 255, # trade name
    125: 255, # microwave
    126: 255, # pot
    127: 11, # animal → person (animate)
    128: 18, # bicycle
    129: 255,  # lake → terrain
    130: 255, # dishwasher
    131: 255, # screen
    132: 255, # blanket
    133: 255, # sculpture
    134: 255, # hood
    135: 5,  # sconce → pole
    136: 255, # vase
    137: 6,  # traffic light
    138: 255, # tray
    139: 255, # ashcan
    140: 255, # fan
    141: 9,  # pier → terrain
    142: 255, # crt screen
    143: 255, # plate
    144: 255, # monitor
    145: 255, # bulletin board
    146: 255, # shower
    147: 255, # radiator
    148: 255, # glass
    149: 255, # clock
    150: 255, # flag
}


def convert_ade20k_to_cityscapes(mask: np.ndarray):
    """Convert ADE20K mask IDs (0–149) → Unified Cityscapes space (0–19)."""
    out = np.zeros_like(mask, dtype=np.int64)
    for k, v in ADE20K_TO_CITYSCAPES.items():
        out[mask == k] = v
    # print("Unique ADE mask before map:", np.unique(mask))
    # print("Unique after map:", np.unique(out))

    return out

# ============================================================
# 🔹 ADE20K Dataset Loader
# ============================================================
class ADE20KDataset(Dataset):
    def __init__(self, root_dir, filter20 = False ,split="train"):
        self.samples = []
        split_dir = {"train": "training", "val": "validation", "test": "validation"}[split]

        self.img_root = os.path.join(root_dir, "images", split_dir)
        self.mask_root = os.path.join(root_dir, "annotations", split_dir)

        if not os.path.exists(self.img_root):
            raise FileNotFoundError(f"❌ Missing folder: {self.img_root}")

        img_files = sorted([f for f in os.listdir(self.img_root) if f.endswith(".jpg")])
        mask_files = sorted([f for f in os.listdir(self.mask_root) if f.endswith(".png")])

        for img_file in img_files:
            base = img_file.replace(".jpg", "")
            seg_file = base + ".png"
            img_path = os.path.join(self.img_root, img_file)
            seg_path = os.path.join(self.mask_root, seg_file)

            # ---  20 Class filtering ---
            if filter20:
                mask_img = Image.open(seg_path)
                mask_rgb = np.array(mask_img.convert("RGB"), dtype=np.uint8)
                mask = ade20k_rgb_to_class(mask_rgb)
                mask = convert_ade20k_to_cityscapes(mask)

                unknown_ratio = np.mean(mask == 255)
                if unknown_ratio > 0.5:
                    continue
                
            if os.path.exists(seg_path):
                self.samples.append((img_path, seg_path))

        print(f"Loaded {len(self.samples)} ADE20K {split} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)

        # decode ADE20K mask
        mask_img = Image.open(mask_path)
        # if mask_img.mode == "I":
        #     mask = np.array(mask_img, dtype=np.int32) % 256
        # else:
        mask_rgb = np.array(mask_img.convert("RGB"), dtype=np.uint8)
        mask = ade20k_rgb_to_class(mask_rgb)

        mask = Image.fromarray(mask.astype(np.uint8)).resize(IMG_SIZE, resample=Image.NEAREST)
        mask = np.array(mask, dtype=np.uint8)
        mask = convert_ade20k_to_cityscapes(mask)

        image = np.array(image).transpose(2, 0, 1) / 255.0

        image = torch.tensor(image, dtype=torch.float32)
        mask = torch.tensor(mask, dtype=torch.long)
        return image, mask, img_path


# ============================================================
# 🔹 Unified MUSES Dataset Loader (All Weather + Day/Night)
# ============================================================
class MusesDataset(Dataset):
    def __init__(self, root_dir, weather="all",split="train"):
        self.root = root_dir
        self.split = split
        self.samples = []

        # sub-conditions ทั้งหมด เช่น clear/day, fog/night, rain/day ...
        all_weathers = ["clear", "fog", "rain", "snow"]
        if weather == "all":
            weathers = all_weathers
        elif isinstance(weather, (list, tuple)):
            weathers = [w for w in weather if w in all_weathers]
        elif weather in all_weathers:
            weathers = [weather]
        else:
            raise ValueError(f"Invalid weather '{weather}'. Choose from {all_weathers + ['all']}")
        # print(f"✅ Loading MUSES dataset with weathers: {weathers}")
        times_of_day = ["day", "night"]

        frame_root = os.path.join(root_dir, "frame_camera", split)
        mask_root = os.path.join(root_dir, "gt_semantic", split)

        for w in weathers:
            for tod in times_of_day:
                img_dir = os.path.join(frame_root, w, tod)
                mask_dir = os.path.join(mask_root, w, tod)

                if not (os.path.exists(img_dir) and os.path.exists(mask_dir)):
                    print(f"⚠️ Skip missing folder: {img_dir} or {mask_dir}")
                    continue

                img_files = sorted([f for f in os.listdir(img_dir) if f.endswith(".png")])
                mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith("_gt_labelTrainIds.png")])

                for img_file in img_files:
                    prefix = img_file.replace("_frame_camera.png", "")
                    match = [m for m in mask_files if prefix in m]
                    if match:
                        self.samples.append((os.path.join(img_dir, img_file),
                                             os.path.join(mask_dir, match[0])))

        print(f"Loaded {len(self.samples)} MUSES {split} samples with weathers: {weathers}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, resample=Image.NEAREST)

        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long), img_path



# ============================================================
# 🔹 Cityscapes Dataset Loader (Final Verified)
# ============================================================
class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split="train"):
        self.split = split
        self.samples = []

        self.img_root = os.path.join(root_dir, "leftImg8bit", split)
        self.mask_root = os.path.join(root_dir, "gtFine", split)

        if not os.path.exists(self.img_root):
            raise FileNotFoundError(f"❌ Missing folder: {self.img_root}")

        for city in os.listdir(self.img_root):
            img_dir = os.path.join(self.img_root, city)
            mask_dir = os.path.join(self.mask_root, city)
            if not os.path.exists(mask_dir):
                continue

            for f in os.listdir(img_dir):
                if f.endswith("_leftImg8bit.png"):
                    base = f.replace("_leftImg8bit.png", "")
                    # ✅ fallback: ถ้าไม่มี labelTrainIds ให้ใช้ labelIds
                    mask_file = None
                    for suffix in ["_gtFine_labelTrainIds.png"] : # , "_gtFine_labelIds.png"]
                        candidate = os.path.join(mask_dir, base + suffix)
                        if os.path.exists(candidate):
                            mask_file = candidate
                            break
                    if mask_file:
                        self.samples.append((os.path.join(img_dir, f), mask_file))

        print(f"Loaded {len(self.samples)} Cityscapes {split} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, 
                                                         resample=Image.NEAREST)
        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)
        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long), img_path


# ============================================================
# 🔹 ACDC Dataset Loader (Multiple Weathers)
# ============================================================
class ACDCDataset(Dataset):
    """
    ACDC dataset loader supporting multiple weathers (fog/night/rain/snow)
    Structure expected:
        dataset/acdc/{rgb,gt}/{weather}/{split}/{video}/
    """
    def __init__(self, root_dir, weather="all", split="train"):
        self.split = split
        self.samples = []

        # --- weather selection ---
        all_weathers = ["fog", "night", "rain", "snow"]
        if weather == "all":
            weathers = all_weathers
        elif isinstance(weather, (list, tuple)):
            weathers = [w for w in weather if w in all_weathers]
        elif weather in all_weathers:
            weathers = [weather]
        else:
            raise ValueError(f"Invalid weather '{weather}'. Choose from {all_weathers + ['all']}")

        # --- collect all samples ---
        for w in weathers:
            rgb_root = os.path.join(root_dir, "rgb_anon", w, split)
            gt_root = os.path.join(root_dir, "gt", w, split)
            if not os.path.exists(rgb_root):
                print(f"⚠️ Skip weather '{w}' (not found: {rgb_root})")
                continue

            for vid in os.listdir(rgb_root):
                img_dir = os.path.join(rgb_root, vid)
                mask_dir = os.path.join(gt_root, vid)
                # if not os.path.isdir(img_dir):
                #     continue

                for f in os.listdir(img_dir):
                    if f.endswith("_rgb_anon.png"):
                        base = f.replace("_rgb_anon.png", "")
                        mask_file = None
                        for suffix in ["_gt_labelTrainIds.png"]: #  "_gt_labelTrainIds.png", "_gt_labelIds.png"
                            candidate = os.path.join(mask_dir, base + suffix)
                            if os.path.exists(candidate):
                                mask_file = candidate
                                break
                        if mask_file:
                            self.samples.append((os.path.join(img_dir, f), mask_file))
                            # print(f"✅ Found mask for image: {mask_file}")
                        else:
                            print(f"⚠️ Missing mask for image: {os.path.join(img_dir, f)}")
                            

        print(f"✅ Loaded {len(self.samples)} ACDC {split} samples from weathers: {weathers}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        # --- Load image + mask ---
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, 
                                                         resample=Image.NEAREST)

        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)
        # # --- Optional transforms ---
        # if self.transform:
        #     image, mask = self.transform(image, mask)

        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long), img_path

class BDD10kDataset(Dataset):
    def __init__(self, root_dir, split="train", onlyimg=False):
        self.split = split
        self.onlyimg = onlyimg
        self.samples = []

        self.img_root = os.path.join(root_dir, "images", "10k", split)
        self.mask_root = os.path.join(root_dir, "labels", split)

        if not os.path.exists(self.img_root):
            raise FileNotFoundError(f"❌ Missing folder: {self.img_root}")

        img_files = sorted([f for f in os.listdir(self.img_root) if f.endswith(".jpg")])
    
        for img_file in img_files:
            base = img_file.replace(".jpg", "")
            seg_file = base + "_train_id.png"
            img_path = os.path.join(self.img_root, img_file)
            seg_path = os.path.join(self.mask_root, seg_file)
                
            if os.path.exists(seg_path):
                self.samples.append((img_path, seg_path))

        print(f"Loaded {len(self.samples)} BDD10K {split} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, 
                                                         resample=Image.NEAREST)
        if self.onlyimg:
            return image, mask
        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)
        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long), img_path

class MVDataset(Dataset):
    def __init__(self, root_dir, split="train", onlyimg=False):
        self.split = split
        self.onlyimg = onlyimg
        self.samples = []

        self.img_root = os.path.join(root_dir, split, "images")
        self.mask_root = os.path.join(root_dir, split, "labels")

        if not os.path.exists(self.img_root):
            raise FileNotFoundError(f"❌ Missing folder: {self.img_root}")

        img_files = sorted([f for f in os.listdir(self.img_root) if f.endswith(".jpg")])
    
        for img_file in img_files:
            base = img_file.replace(".jpg", "")
            seg_file = base + ".png"
            img_path = os.path.join(self.img_root, img_file)
            seg_path = os.path.join(self.mask_root, seg_file)
                
            if os.path.exists(seg_path):
                #########################################################################
    
                #########################################################################
                self.samples.append((img_path, seg_path))

        print(f"Loaded {len(self.samples)} Mapillary Vistas {split} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        mask = Image.open(mask_path).convert("L").resize(IMG_SIZE, 
                                                         resample=Image.NEAREST)
        if self.onlyimg:
            return image, mask
        image = np.array(image).transpose(2, 0, 1) / 255.0
        mask = np.array(mask, dtype=np.uint8)
        
        return torch.tensor(image, dtype=torch.float32), torch.tensor(mask, dtype=torch.long), img_path


### Dataset Loader

In [5]:
from torch.utils.data import DataLoader, ConcatDataset

BATCH_SIZE = 1
print("Full workesrs:", os.cpu_count())
mynum_workers = os.cpu_count() // 3
print(f"Using {mynum_workers} workers for DataLoader")

# ============================================================
# LOAD DATASETS ทั้งหมด (train / val / test)
# ============================================================

print('\nValidation Dataset Loading ...')
muse_val_clear  = MusesDataset(muse_dir, weather="clear", split="val")
muse_val_fog    = MusesDataset(muse_dir, weather="fog", split="val")
muse_val_rain   = MusesDataset(muse_dir, weather="rain", split="val")
muse_val_snow   = MusesDataset(muse_dir, weather="snow", split="val")
ade_val         = ADE20KDataset(ade20k_dir, split="val")
cs_val          = CityscapesDataset(cs_dir, split="val")  
acdc_val_fog    = ACDCDataset(acdc_dir, weather="fog", split="val")
acdc_val_night  = ACDCDataset(acdc_dir, weather="night",   split="val")
acdc_val_rain   = ACDCDataset(acdc_dir, weather="rain",   split="val")
acdc_val_snow   = ACDCDataset(acdc_dir, weather="snow",   split="val")
bdd_val         = BDD10kDataset(bdd_dir, split="val")
mv_val          = MVDataset(mv_dir, split="val")


all_val = [muse_val_clear,
           muse_val_fog,  
           muse_val_rain, 
           muse_val_snow, 

           ade_val,       
           cs_val,        

           acdc_val_fog,  
           acdc_val_night,
           acdc_val_rain, 
           acdc_val_snow,

           bdd_val,
           mv_val,
        ]


# ============================================================
# รวม dataset เข้าด้วยกัน
# ============================================================
val_dataset   = ConcatDataset(all_val)
# test_dataset  = ConcatDataset([muses_test, ade_test])
print('\nTotal Dataset Loaded')

def create_dataloader(validateaset = None):
    if validateaset is None:
        # ============================================================
        # สร้าง Full DataLoader
        # ============================================================
        val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, 
                                shuffle=False, num_workers=mynum_workers)
        return val_loader
    val_loader   = DataLoader(validateaset, batch_size=BATCH_SIZE, 
                            shuffle=False, num_workers=mynum_workers)
    return val_loader

# Full DataLoader
val_loader = create_dataloader() 

muse_val_clear  = create_dataloader(muse_val_clear  )
muse_val_fog    = create_dataloader(muse_val_fog    )
muse_val_rain   = create_dataloader(muse_val_rain   )
muse_val_snow   = create_dataloader(muse_val_snow   )
ade_val         = create_dataloader(ade_val         )
cs_val          = create_dataloader(cs_val          )
acdc_val_fog    = create_dataloader(acdc_val_fog    )
acdc_val_night  = create_dataloader(acdc_val_night  )
acdc_val_rain   = create_dataloader(acdc_val_rain   )
acdc_val_snow   = create_dataloader(acdc_val_snow   )
bdd_val         = create_dataloader(bdd_val         )
mv_val          = create_dataloader(mv_val          )
# ============================================================
# แสดงสรุปผล
# ============================================================
print(f"✅ Val:   {len(val_dataset)} samples")

Full workesrs: 72
Using 24 workers for DataLoader

Validation Dataset Loading ...
Loaded 75 MUSES val samples with weathers: ['clear']
Loaded 58 MUSES val samples with weathers: ['fog']
Loaded 59 MUSES val samples with weathers: ['rain']
Loaded 58 MUSES val samples with weathers: ['snow']
Loaded 838 ADE20K val samples
Loaded 500 Cityscapes val samples
✅ Loaded 100 ACDC val samples from weathers: ['fog']
✅ Loaded 106 ACDC val samples from weathers: ['night']
✅ Loaded 100 ACDC val samples from weathers: ['rain']
✅ Loaded 100 ACDC val samples from weathers: ['snow']
Loaded 1000 BDD10K val samples
Loaded 2000 Mapillary Vistas val samples

Total Dataset Loaded
✅ Val:   4994 samples


## Model

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

criterion = nn.CrossEntropyLoss(ignore_index=255)
class CLIPSegBaseline(nn.Module):
    def __init__(self, clip_model, num_classes=NUM_CLASSES):
        super().__init__()
        self.clip = clip_model.vision_model  # HF: CLIPVisionModel
        self.num_classes = num_classes
        
        # Freeze CLIP encoder
        for param in self.clip.parameters():
            param.requires_grad = False
        
        self.feature_dim = 1024  # usually 768
        
        # Decoder head
        self.decoder = nn.Sequential(
            nn.Conv2d(self.feature_dim, 512, 3, padding=1),
            nn.ReLU(),
            nn.Dropout2d(p=0.1),
            nn.Conv2d(512, 256, 3, padding=1),
            nn.ReLU(),
            nn.Dropout2d(p=0.05),
            nn.Conv2d(256, num_classes, 1)
        )
    
    def forward(self, x):
        B, _, H, W = x.shape
        seq_len = (H // self.clip.config.patch_size) * (W // self.clip.config.patch_size)  # 14*14=196

        with torch.no_grad():
            # Patch embedding
            x_vit = self.clip.embeddings.patch_embedding(x)  # [B, 768, 14, 14]
            x_vit = x_vit.flatten(2).transpose(1, 2)        # [B, 196, 768]

            # CLS token
            if hasattr(self.clip.embeddings, "cls_token"):
                cls_token = self.clip.embeddings.cls_token.expand(B, -1, -1)
            else:
                cls_token = torch.zeros(B, 1, self.feature_dim, device=x.device)
            x_vit = torch.cat([cls_token, x_vit], dim=1)  # [B, 197, 768]

            # Positional embeddings: index
            pos_ids = torch.arange(x_vit.shape[1], device=x.device)
            pos_embeds = self.clip.embeddings.position_embedding(pos_ids)  # [197, 768]
            x_vit = x_vit + pos_embeds.unsqueeze(0)  # broadcast to [B, 197, 768]

            # LayerNorm
            x_vit = self.clip.pre_layrnorm(x_vit)    
            # x_vit = self.clip.embeddings.ln_pre(x_vit)

            # Transformer
            x_vit = self.clip.encoder(x_vit)[0]
            features = x_vit[:, 1:, :].float()  # skip CLS token

        
        # Reshape spatial map 14x14
        features = features.reshape(B, H // self.clip.config.patch_size, W // self.clip.config.patch_size, self.feature_dim)
        features = features.permute(0, 3, 1, 2)  # [B, 768, 14, 14]
        
        # Decode
        seg_logits = self.decoder(features)  # [B, num_classes, 14, 14]
        
        # Upsample
        seg_logits = F.interpolate(seg_logits, size=(H, W), 
                                   mode='bilinear', align_corners=False)
        return seg_logits


### Call base Model

In [7]:
def call_model(): 
    # ----- Reload base CLIP seg model to terminate Block-LoRA block -----
    clip_model = CLIPModel.from_pretrained(clip32_dir, use_safetensors=True)
    
    base_model = CLIPSegBaseline(
        clip_model,
        num_classes=NUM_CLASSES,
    ).to(DEVICE)
    
    if "pth" in MODEL_PATH:
        state_dict = torch.load(MODEL_PATH, 
                        map_location=DEVICE,
                        weights_only=False)["model_state_dict"] # True
    else:
        state_dict = torch.load(MODEL_PATH, 
                        map_location=DEVICE,
                        weights_only=False) # True
    
    decoder_state = {k.replace('decoder.', ''): v for k, v in state_dict.items() if 'decoder' in k}
    base_model.decoder.load_state_dict(decoder_state, strict=False)

    # # 🔧 เพิ่มความเร็วด้วย torch.compile() และ channels_last
    # base_model = torch.compile(base_model)
    # base_model.to(memory_format=torch.channels_last)
    # # base_model.to(memory_format=torch.contiguous_format)  

    print("Truly Reload Base Model")
    return base_model

## Validate Model

In [8]:

# from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.notebook import tqdm

@torch.no_grad()
def validate_oracle(model, peft_model, val_loader, use_amp=True, lora_type = "LoRA"):
    losses = AverageMeter()
    mious = AverageMeter()
    
    pbar = tqdm(val_loader, desc="Validation")
    weights = {}

    model.eval()
    peft_model.set_adapter(current_target_domain_name)
    for images, masks, img_path in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE).long()
        
        if lora_type == "LoRA":
            model.clip = peft_model
        else:
            model.clip = peft_model.model
            model.to(DEVICE)
        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                # loss = criterion(outputs, masks)
        else:
            outputs = model(images)
            # loss = criterion(outputs, masks)
        
        pred = torch.argmax(outputs, dim=1)
        
        # losses.update(loss.item(), images.size(0))
        miou = calculate_miou(pred, masks, NUM_CLASSES)
        mious.update(miou, images.size(0))

        pbar.set_postfix({'mIoU': f'{mious.avg:.4f}'})
    
    print("Average mIoU:", mious.avg)
    return mious, weights

@torch.no_grad()
def validate_uniform(model, peft_model, val_loader, use_amp=True, lora_type = "LoRA"):
    losses = AverageMeter()
    mious = AverageMeter()
    
    pbar = tqdm(val_loader, desc="Validation")
    weights = {}

    model.eval()
    weight_dict, merged_adpater_name = merge(
                    peft_model=peft_model,
                    target_domain=current_target_domain_name,
                    remove_target_adapter=True,
                    mode="uniform", #
                    combination_type=combination_type,
                )
    
    for domain, weight in weight_dict.items():
        weights.setdefault(domain, []).append(weight)

    for images, masks, img_path in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE).long()
        
        if lora_type == "LoRA":
            model.clip = peft_model
        else:
            model.clip = peft_model.model
            model.to(DEVICE)
        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                # loss = criterion(outputs, masks)
        else:
            outputs = model(images)
            # loss = criterion(outputs, masks)
        
        pred = torch.argmax(outputs, dim=1)
        
        # losses.update(loss.item(), images.size(0))
        miou = calculate_miou(pred, masks, NUM_CLASSES)
        mious.update(miou, images.size(0))
        
        pbar.set_postfix({'mIoU': f'{mious.avg:.4f}'})

    peft_model.set_adapter("ACDC-fog")
    peft_model.delete_adapter(merged_adpater_name)
    print("Average mIoU:", mious.avg)
    return mious, weights


@torch.no_grad()
def validate_semla(model, peft_model, val_loader, use_amp=True, lora_type = "LoRA"):
    losses = AverageMeter()
    mious = AverageMeter()
    
    pbar = tqdm(val_loader, desc="Validation")
    weights = {}

    model.eval()
    for images, masks, img_path in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE).long()
        
        current_embedding = embedding_manager.embed_image(img_path[0])
        # Mulnipulate adapter weight base on 'current_embedding'
        weight_dict, merged_adpater_name = merge(
                            peft_model=peft_model,
                            target_domain=current_target_domain_name,
                            remove_target_adapter=True,
                            mode="centroid", 
                            target_embedding=current_embedding,
                            softmax_temperature=temperature,
                            top_k=top_k,
                            combination_type=combination_type,
                            similarity_measure=NAME_MEASURE_MAPPING[similarity_measure_name],
                            sort_descending=True,
                        )
    
        for domain, weight in weight_dict.items():
            weights.setdefault(domain, []).append(weight)
        
        if lora_type == "LoRA":
            model.clip = peft_model
        else:
            model.clip = peft_model.model
            model.to(DEVICE)
            
        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                # loss = criterion(outputs, masks)
        else:
            outputs = model(images)
            # loss = criterion(outputs, masks)
        
        pred = torch.argmax(outputs, dim=1)
        
        # losses.update(loss.item(), images.size(0))
        miou = calculate_miou(pred, masks, NUM_CLASSES)
        mious.update(miou, images.size(0))
        
        peft_model.set_adapter("ACDC-fog")
        peft_model.delete_adapter(merged_adpater_name)
        pbar.set_postfix({'mIoU': f'{mious.avg:.4f}'})
    
    print("Average mIoU:", mious.avg)
    return mious, weights

@torch.no_grad()
def validate(model, val_loader, use_amp=True):
    model.eval()
    # losses = AverageMeter()
    mious = AverageMeter()
    
    pbar = tqdm(val_loader, desc="Validation")
    
    for images, masks, img_path in pbar:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE).long()
        
        if use_amp:
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, masks)
        else:
            outputs = model(images)
            loss = criterion(outputs, masks)
        
        pred = torch.argmax(outputs, dim=1)
        
        # losses.update(loss.item(), images.size(0))
        miou = calculate_miou(pred, masks, NUM_CLASSES)
        mious.update(miou, images.size(0))
        
        pbar.set_postfix({'mIoU': f'{mious.avg:.4f}'})
    
    return mious.avg # losses.avg,

class AverageMeter:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def calculate_miou(pred, target, num_classes=20, ignore_index=255):
    pred = pred.cpu().numpy()
    target = target.cpu().numpy()
    
    valid = (target != ignore_index)
    pred = pred[valid]
    target = target[valid]
    
    if len(pred) == 0:
        return 0.0
    
    ious = []
    present_classes = np.unique(target)
    
    for cls in present_classes:
        if cls == ignore_index:
            continue
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = np.logical_and(pred_cls, target_cls).sum()
        union = np.logical_or(pred_cls, target_cls).sum()
        if union > 0:
            ious.append(intersection / union)
    
    return np.mean(ious) if ious else 0.0

def validate_model(model, val_loader):
    val_loss, val_miou = validate(model, val_loader)
    print(f"\n📊 Validation Summary:")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val mIoU: {val_miou:.4f}")


## helper function from SemLA

In [9]:
# Define distance measure mappings
NAME_MEASURE_MAPPING = {
    "euclidean": lambda u, v: 1. / scipy.spatial.distance.euclidean(u.squeeze(), v.squeeze()),
    "cosine": lambda u, v: scipy.spatial.distance.cosine(u.squeeze(), v.squeeze()),
}

def load_domains_from_yaml(file_path: str) -> List[str]:
    """Load domains from a YAML file."""
    with open(file_path, 'r') as f:
        return yaml.safe_load(f)

def load_config_from_yaml(file_path: str) -> Dict[str, Any]:
    """Load configuration parameters from a YAML file."""
    with open(file_path, 'r') as f:
        return yaml.safe_load(f)

def save_results(results: Dict, weights: Optional[Dict] = None, output_dir: str = "./results") -> None:
    """Save results and weights to JSON files."""
    
    # Change the current working directory to the root directory
    root_dir = os.path.abspath(os.path.join(os.path.dirname(__file__)))
    print(f"Changing current working directory to {root_dir}")
    os.chdir(root_dir)
    
    os.makedirs(output_dir, exist_ok=True)
    
    with open(os.path.join(output_dir, "results.json"), 'w') as f:
        json.dump(results, f, indent=4)
    
    if weights is not None:
        with open(os.path.join(output_dir, "weights.json"), 'w') as f:
            json.dump(weights, f, indent=4)
    
    print(f"Results saved to {output_dir}")

def benchmark_zeroshot(source_domains: List[str], target_domains: List[str], 
                      output_dir: str) -> None:
    """Run zero-shot benchmark experiment."""
    orchestrator = DomainOrchestrator(source_domains)
    results = orchestrator.benchmark_zeroshot(target_domains)
    save_results(results, output_dir=output_dir)

def benchmark_oracle(source_domains: List[str], target_domains: List[str], 
                    output_dir: str) -> None:
    """Run oracle benchmark experiment."""
    orchestrator = DomainOrchestrator(domains=source_domains)
    results = orchestrator.benchmark_oracle(target_domains=target_domains)
    save_results(results, output_dir=output_dir)

def uniform_merge(source_domains: List[str], target_domains: List[str], 
                 remove_target_adapter: bool, output_dir: str) -> None:
    """Run uniform merge experiment."""
    orchestrator = DomainOrchestrator(domains=source_domains)
    results, weights = orchestrator.benchmark_uniform(
        target_domains=target_domains,
        remove_target_adapter=remove_target_adapter,
    )
    save_results(results, weights, output_dir=output_dir)

def semla_merge(source_domains: List[str], target_domains: List[str], 
                config: Dict[str, Any], remove_target_adapter: bool, 
                output_dir: str) -> None:
    """Run online merge experiment."""
    similarity_measure_name = config.get("similarity_measure_name", "euclidean")
    temperature = config.get("temperature", 0.05)
    top_k = config.get("top_k", 5)
    combination_type = config.get("combination_type", "cat")
    
    orchestrator = DomainOrchestrator(source_domains)
    results, weights = orchestrator.benchmark_semla(
        target_domains=target_domains,
        remove_target_adapter=remove_target_adapter,
        similarity_measure=NAME_MEASURE_MAPPING[similarity_measure_name],
        softmax_temperature=temperature,
        top_k=top_k,
        combination_type=combination_type
    )
    save_results(results, weights, output_dir=output_dir)

def parse_args():
    """Parse command line arguments."""
    parser = argparse.ArgumentParser(description="Domain adaptation experiments")
    
    # Required arguments
    parser.add_argument("--experiment", type=str, required=True, 
                        choices=["zeroshot", "oracle", "uniform", "semla"],
                        help="Type of experiment to run")
    
    # Optional arguments with defaults
    parser.add_argument("--source_domains", type=str, 
                        help="Path to YAML file containing source domains")
    parser.add_argument("--target_domains", type=str, 
                        help="Path to YAML file containing target domains")
    parser.add_argument("--semla_config", type=str, 
                        help="Path to YAML file containing configuration parameters")
    parser.add_argument("--output_dir", type=str, default="./results",
                        help="Directory to save results")
    parser.add_argument("--remove_target_adapter", action="store_true", 
                        help="Whether to remove target adapter")
    
    return parser.parse_args()

## Merge Adapter

In [10]:
def merge(
        peft_model: None,
        target_domain: str,
        remove_target_adapter: bool,
        mode: list[str],                     #Literal["uniform", "centroid"],
        target_embedding=None,
        softmax_temperature: int | None = 0.05,
        top_k: int = 4,  # number of domains to merge Originally = 5
        combination_type: str = "cat",
        similarity_measure: Callable[
            [npt.NDArray, npt.NDArray], np.float64] = lambda v1, v2: np.linalg.norm(v1 - v2),
        sort_descending: bool = True,
        visu: bool = False,

    ) -> tuple[dict[str, float], str]:
    """
    Merge the source domains and benchmark the merged adapter on the target domain.
    """
    new_source_domains = []
    if remove_target_adapter:
        print(f"Removing {target_domain} from source domains!") if visu else None
        for domain in source_domains:
            if domain != target_domain: 
                new_source_domains.append(domain)

    else:
        new_source_domains = [
            domain
            for domain in source_domains
        ]
    print(new_source_domains) if visu else None

    if mode == "uniform":
        weights = [1 / len(new_source_domains) for _ in range(len(new_source_domains))]
        domain_weight_mapping = {domain: weight for domain, weight in zip(new_source_domains, weights)}

        merged_name = ""
        for n, w in domain_weight_mapping.items():
            merged_name += f"_{n}_{str(w).replace('.','_')}"
        merged_name += f"_{combination_type}_{target_domain}" # Create a unique name for merged adapter so that it does not override existing adapters

        merge_adapters(
            peft_model=peft_model,
            merge_domains=[domain for domain in new_source_domains],
            weights=weights,
            merged_name=merged_name,
            combination_type=combination_type
        )

    elif mode == "centroid":

        similarity_mapping = calculate_similarity_to_domains(
            embedding=target_embedding,
            domains=new_source_domains,  
            similarity_measure=similarity_measure,
            sort_descending=sort_descending
        )

        k_closest_names = list(similarity_mapping.keys())[: top_k]
        k_closest_similarities = list(similarity_mapping.values())[: top_k]

        print(f"\nSimilarities to {top_k} closest domains: ") if visu else None
        for n, d in zip(k_closest_names, k_closest_similarities):
            print(f"{n}: {d}", end=", \n") if visu else None
        print("") if visu else None

        weights = calculate_adapter_weights(k_closest_similarities, softmax_temperature)

        domain_weight_mapping = {
            k_closest_name: weight
            for k_closest_name, weight in zip(k_closest_names, weights)
        }

        merged_name = ""
        for n, w in domain_weight_mapping.items():
            merged_name += f"_{n}_{str(w).replace('.','_')}"
        merged_name += f"_{combination_type}_{target_domain}" # Create a unique name for merged adapter so that it does not override existing adapters

        merge_adapters(
            peft_model=peft_model,
            merge_domains=k_closest_names,
            weights=weights,
            merged_name=merged_name,
            combination_type=combination_type
        )

    # print(f"Setting {merged_name} as the active adapter.\n")
    peft_model.set_adapter(merged_name)

    return domain_weight_mapping, merged_name


def merge_adapters(
    peft_model: None,
    merge_domains: list[str],
    weights: list[float],
    merged_name: str,
    combination_type: str,
    visu: bool = False,
) -> None:
    """
    Merge the specified adapters with the specified weights.
    """
    
    if visu:
        print(f"Merging domains with weights:")
        for n, w in zip(merge_domains, weights):
            print(f"{n}: {w}", end=", ")    
        print("")

    peft_model.add_weighted_adapter(
        merge_domains,
        weights,
        merged_name,
        combination_type=combination_type,
    )

# Ok
def calculate_similarity_to_domains(
    embedding: npt.NDArray, 
    domains: None,
    similarity_measure: Callable[[npt.NDArray, npt.NDArray], np.float64],
    sort_descending: bool = True
) -> Dict[str, float]:
    """
    Calculate the similarity between the target embedding and the domain prototypes.
    """
    similarities = []

    for domain in domains:
        prot = source_domains_dict[domain][1]
        similarity = similarity_measure(embedding, prot)
        similarities.append([domain, similarity])

    # sort similarities from lowest to highest
    similarities_dict = dict(sorted(similarities, key=lambda x: x[1], reverse=sort_descending))
    return similarities_dict

# Ok
def load_stat(domain_name):
    domain_path = blora_library_path / Path(domain_name)
    suffix = "_statistics.npz"
    statistics_path = domain_path / f"{domain_name}{suffix}"
    # stats_dict = {}

    # print(f"Statistics file: {statistics_path}")
    # if statistics_path.exists():  # Load the data if it exists
    # print(f"Loading statistics from {domain_name}{suffix} ...")
    stats = np.load(statistics_path)["train_average_embedding"]
    # stats_dict.update({
    #     "train_average_embedding": stats["train_average_embedding"],
    # })
    print(f"Statistics loaded from {domain_name}{suffix}")
    return stats

# Ok
def softmax(x: list[float], softmax_temperature) -> np.ndarray:
    """Compute softmax values for each sets of scores in x."""
    
    # Add error handling for division by zero
    if softmax_temperature == 0:
        softmax_temperature = 1e-6
    exp_x = np.exp(np.divide(x, softmax_temperature))
    return exp_x / np.sum(exp_x, axis=0)

# Ok
def calculate_adapter_weights(similarities:list[float], temperature: float) -> list[float]:
    """
    Calculate the weights for the merged adapter based on the similarities to the source domains.
    """
    weights = softmax(similarities, temperature)
    return weights

## Load Stat: Dataset Centroid

In [11]:
DOMAIN_TO_VAL ={
        'ACDC-fog'    :  acdc_val_fog,
        'ACDC-night'  :  acdc_val_night,
        'ACDC-rain'   :  acdc_val_rain,  
        'ACDC-snow'   :  acdc_val_snow,  
        'ADE20K'      :  ade_val,        
        'CS'          :  cs_val,        
        'MUSES-clear' :  muse_val_clear,        
        'MUSES-fog'   :  muse_val_fog,      
        'MUSES-rain'  :  muse_val_rain,     
        'MUSES-snow'  :  muse_val_snow,         
        'BDD'         :  bdd_val,   
        'MV'          :  mv_val,    
}

In [12]:
# Load source domains
source_domains = load_domains_from_yaml(source_domains_dir) 
source_domains_dict = {}
for domain in source_domains:
    source_domains_dict[domain] = [ # 0.) LoRA path 
                                    # 1.) Centroid Embedding
                                    # 2.) Validate Set Loader
                                    # 3.) BLoRA path
                                   Path(os.path.join(lora_library_path, domain)),
                                   load_stat(domain),
                                   DOMAIN_TO_VAL[domain],
                                   Path(os.path.join(blora_library_path, domain)),
                                   ]

# Load target domains
target_domains = load_domains_from_yaml(target_domains_dir) 

# Load config if provided
semla_config = load_config_from_yaml(semla_config_dir)
similarity_measure_name = semla_config.get("similarity_measure_name", "euclidean")

Statistics loaded from ACDC-fog_statistics.npz
Statistics loaded from ACDC-night_statistics.npz
Statistics loaded from ACDC-rain_statistics.npz
Statistics loaded from ACDC-snow_statistics.npz
Statistics loaded from ADE20K_statistics.npz
Statistics loaded from CS_statistics.npz
Statistics loaded from MUSES-clear_statistics.npz
Statistics loaded from MUSES-fog_statistics.npz
Statistics loaded from MUSES-rain_statistics.npz
Statistics loaded from MUSES-snow_statistics.npz
Statistics loaded from BDD_statistics.npz
Statistics loaded from MV_statistics.npz


## WorkStation

In [13]:
# Cell 3: create embedding manager
embedding_manager = EmbeddingManager()
print("EmbeddingManager created:", embedding_manager)
img_path = Path("/home/chenyinjia/data/dataset/muses/frame_camera/train/snow/night/REC0066_frame_500898_frame_camera.png")


# quick device / model status check (optional)
print(img_path)
emb = embedding_manager.embed_image(img_path)
print("Sample embedding shape:", np.asarray(emb).shape)

EmbeddingManager created: <embedding.EmbeddingManager object at 0x7f24f03bf100>
/home/chenyinjia/data/dataset/muses/frame_camera/train/snow/night/REC0066_frame_500898_frame_camera.png


Sample embedding shape: (1, 768)


In [14]:
# print(similarity_measure_name)

# a = merge(
#       peft_model=None,
#       target_domain="MUSES-snow", 
#       remove_target_adapter=True, 
#       mode='centroid',
#       similarity_measure=NAME_MEASURE_MAPPING[similarity_measure_name],
#       target_embedding=emb
#       )

In [15]:
# model2 = call_model()

# blr_path = os.path.join(blora_library_path, "CS")
# peft_model= BlockLoraModel.from_pretrained(model2.clip,  blr_path)

In [16]:
# model2 = call_model()
# peft_model2 = set_current_target_domain(model2.clip, lora_type="blora")

In [17]:
# weight_dict, merged_adpater_name = merge(
#                             peft_model=peft_model2,
#                             target_domain="CS",
#                             remove_target_adapter=True,
#                             mode="uniform", #
#                             combination_type="cat",
#                             visu=True
#                         )

## Common Load      

In [18]:
def set_current_target_domain(clip_model, lora_type="LoRA"):
    """
    Set the current target domain to the specified domain.
    """
    # We need to load all adapters each time the target domain changes because the same
    # config can't be used across datasets and PEFT does not allow us to change the base
    # model and keep the loaded adapters

    # print(f"Setting current target domain to {target_domain}.\n")
    print("Log-In All LoRAs Library")

    clip_model = load_adapters(clip_model, lora_type)
    return clip_model

def load_adapters(clip_model, lora_type):
    """
    Load all adapters for the source domains.
    """
    FirstLoRA = True
    for source_domain in source_domains:
        lora_path = source_domains_dict[source_domain][0] if lora_type == "LoRA" else source_domains_dict[source_domain][3]
        assert lora_path.exists(), lora_path
        # print(f"Loading LoRA: '{lora_path}' ...")
        clip_model, FirstLoRA   = load_lora(clip_model, 
                                            source_domain, 
                                            lora_path,
                                            FirstLoRA,
                                            lora_type)
        print(f"{source_domain[0:7]} LoRA: \t'{lora_path}' loaded")
    return clip_model

def load_lora(clip_model,
              domain_name, 
              lora_path,
              FirstLoRA = True,
              lora_type = "LoRA"):
    """
    Load the LoRA adapter for the specified domain.
    """
    if FirstLoRA:
        # Wrap the model in PeftModel class the first time an adapter is loaded
        # The base model should be loaded with target domain config to avoid label space mismatch
        if lora_type == "LoRA":
            clip_model = PeftModel.from_pretrained(
                clip_model, 
                lora_path, 
                domain_name
            )
        else:
            clip_model = BlockLoraModel.from_pretrained(
                clip_model, 
                lora_path, 
            )
        clip_model.load_adapter(lora_path, domain_name)
        print("Load First LoRA adapter:", domain_name)
        FirstLoRA = False
    else:
        clip_model.load_adapter(lora_path, domain_name)
    
    return clip_model, FirstLoRA

# Experiment on LoRA

## TTA

In [19]:
# similarity_measure_name = semla_config.get("similarity_measure_name", "euclidean")
# temperature = semla_config.get("temperature", 0.05)
# top_k = semla_config.get("top_k", 5)
# combination_type = semla_config.get("combination_type", "cat")

# # Call base model
# model3 = call_model()

# # Gen LoRA Model and load all adapters
# peft_model = set_current_target_domain(model3.clip)
# tta_path = os.path.join(output_dir, "result_LoRA", "TTA")
# os.makedirs(tta_path, exist_ok = True)

# for current_target_domain_name in target_domains[:]:
#     print(current_target_domain_name)

#     t0 = time.time()
#     mIoU, weights = validate_semla( model3, 
#                                     peft_model,
#                                     source_domains_dict[current_target_domain_name][2]
#                                         )
#     # convert numpy.float64 เป็น float ก่อน
#     total = time.time() - t0

#     print(current_target_domain_name, float(mIoU.avg))
#     data_to_save = {
#         "Target": current_target_domain_name,
#         "mIoU": float(mIoU.avg),
#         "Time": total,
#         "weights": weights,
        
#     }
#     savepath = os.path.join(tta_path, f"{current_target_domain_name}.json")
#     with open(savepath, "w") as f:
#         json.dump(data_to_save, f, indent=4)
    


## UNIFORM


In [20]:
# combination_type = semla_config.get("combination_type", "cat")

# # Call base model
# model3 = call_model()

# # Gen LoRA Model and load all adapters
# peft_model = set_current_target_domain(model3.clip)
# uniform_path = os.path.join(output_dir, "result_LoRA", "UNI")
# os.makedirs(uniform_path, exist_ok = True)

# for current_target_domain_name in target_domains[:]:
#     print(current_target_domain_name)

#     t0 = time.time()
#     mIoU, weights   = validate_uniform( model3, 
#                                         peft_model,
#                                         source_domains_dict[current_target_domain_name][2]
#                                         )
#     # convert numpy.float64 เป็น float ก่อน
#     total = time.time() - t0

#     print(current_target_domain_name, float(mIoU.avg))
#     data_to_save = {
#         "Target": current_target_domain_name,
#         "mIoU": float(mIoU.avg),
#         "Time": total,
#         "weights": weights,
        
#     }
#     savepath = os.path.join(uniform_path, f"{current_target_domain_name}.json")
#     with open(savepath, "w") as f:
#         json.dump(data_to_save, f, indent=4)

## Oracle

In [21]:
# # Call base model
# model3 = call_model()

# # Gen LoRA Model and load all adapters
# peft_model = set_current_target_domain(model3.clip)
# oracle_path = os.path.join(output_dir, "result_LoRA", "ORA")
# os.makedirs(oracle_path, exist_ok = True)

# for current_target_domain_name in target_domains[:]:
#     print(current_target_domain_name)

#     t0 = time.time()
#     mIoU, weights   = validate_oracle( model3, 
#                                         peft_model,
#                                         source_domains_dict[current_target_domain_name][2]
#                                         )
#     # convert numpy.float64 เป็น float ก่อน
#     total = time.time() - t0

#     print(current_target_domain_name, float(mIoU.avg))
#     data_to_save = {
#         "Target": current_target_domain_name,
#         "mIoU": float(mIoU.avg),
#         "Time": total,
#         "weights": weights,
        
#     }
#     savepath = os.path.join(oracle_path, f"{current_target_domain_name}.json")
#     with open(savepath, "w") as f:
#         json.dump(data_to_save, f, indent=4)

## Zero Shot

In [22]:
# model3 = call_model()

# zs_path = os.path.join(output_dir, "result_LoRA", "ZS")
# os.makedirs(zs_path, exist_ok = True)

# for current_target_domain_name in target_domains[:]:
#     print(current_target_domain_name)

#     t0 = time.time()
#     mIoU   = validate(model3,  source_domains_dict[current_target_domain_name][2])
#     # convert numpy.float64 เป็น float ก่อน
#     total = time.time() - t0

#     print(current_target_domain_name, float(mIoU))
#     data_to_save = {
#         "Target": current_target_domain_name,
#         "mIoU": float(mIoU),
#         "Time": total,
        
#     }
#     savepath = os.path.join(zs_path, f"{current_target_domain_name}.json")
#     with open(savepath, "w") as f:
#         json.dump(data_to_save, f, indent=4)

# Experiment on Block-LoRA

## TTA

In [23]:
similarity_measure_name = semla_config.get("similarity_measure_name", "euclidean")
temperature = semla_config.get("temperature", 0.05)
top_k = semla_config.get("top_k", 5)
combination_type = semla_config.get("combination_type", "cat")

# Call base model
model3 = call_model()

# Gen LoRA Model and load all adapters
peft_model = set_current_target_domain(model3.clip, lora_type="blora")
tta_path = os.path.join(output_dir, "result_BLoRA", "TTA")
os.makedirs(tta_path, exist_ok = True)

for current_target_domain_name in target_domains[:]:
    print(current_target_domain_name)

    t0 = time.time()
    mIoU, weights = validate_semla(model3, 
                                         peft_model,
                                         source_domains_dict[current_target_domain_name][2],
                                         lora_type = "BLoRA",
                                        )
    # convert numpy.float64 เป็น float ก่อน
    total = time.time() - t0

    print(current_target_domain_name, float(mIoU.avg))
    data_to_save = {
        "Target": current_target_domain_name,
        "mIoU": float(mIoU.avg),
        "Time": total,
        "weights": weights,
        
    }
    savepath = os.path.join(tta_path, f"{current_target_domain_name}.json")
    with open(savepath, "w") as f:
        json.dump(data_to_save, f, indent=4)
    


Truly Reload Base Model


Log-In All LoRAs Library


Load First LoRA adapter: ACDC-fog
ACDC-fo LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-fog' loaded
ACDC-ni LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-night' loaded


ACDC-ra LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-rain' loaded
ACDC-sn LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-snow' loaded


ADE20K LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ADE20K' loaded
CS LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/CS' loaded


...

Average mIoU: 0.3178706365897426
MUSES-snow 0.3178706365897426
BDD


Validation:   0%|          | 0/1000 [00:00<?, ?it/s]

Average mIoU: 0.4595889963056937
BDD 0.4595889963056937
MV


Validation:   0%|          | 0/2000 [00:00<?, ?it/s]

Average mIoU: 0.5135805886624941
MV 0.5135805886624941


## Uniform

In [24]:
combination_type = semla_config.get("combination_type", "cat")

# Call base model
model3 = call_model()

# Gen LoRA Model and load all adapters
peft_model = set_current_target_domain(model3.clip, lora_type="blora")
uniform_path = os.path.join(output_dir, "result_BLoRA", "UNI")
os.makedirs(uniform_path, exist_ok = True)

for current_target_domain_name in target_domains[:]:
    print(current_target_domain_name)

    t0 = time.time()
    mIoU, weights   = validate_uniform( model3, 
                                        peft_model,
                                        source_domains_dict[current_target_domain_name][2],
                                        lora_type = "BLoRA",
                                        )
    # convert numpy.float64 เป็น float ก่อน
    total = time.time() - t0

    print(current_target_domain_name, float(mIoU.avg))
    data_to_save = {
        "Target": current_target_domain_name,
        "mIoU": float(mIoU.avg),
        "Time": total,
        "weights": weights,
        
    }
    savepath = os.path.join(uniform_path, f"{current_target_domain_name}.json")
    with open(savepath, "w") as f:
        json.dump(data_to_save, f, indent=4)

Truly Reload Base Model
Log-In All LoRAs Library
Load First LoRA adapter: ACDC-fog
ACDC-fo LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-fog' loaded
ACDC-ni LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-night' loaded
ACDC-ra LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-rain' loaded


ACDC-sn LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-snow' loaded
ADE20K LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ADE20K' loaded
CS LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/CS' loaded
MUSES-c LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-clear' loaded
MUSES-f LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-fog' loaded
MUSES-r LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-rain' loaded


MUSES-s LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-snow' loaded
BDD LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/BDD' loaded


MV LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MV' loaded
ACDC-fog


Validation:   0%|          | 0/100 [00:00<?, ?it/s]

...

Average mIoU: 0.30879047580318764
MUSES-snow 0.30879047580318764
BDD


Validation:   0%|          | 0/1000 [00:00<?, ?it/s]

Average mIoU: 0.45704501995778624
BDD 0.45704501995778624
MV


Validation:   0%|          | 0/2000 [00:00<?, ?it/s]

Average mIoU: 0.5122802993406422
MV 0.5122802993406422


## Oracle

In [25]:
# Call base model
model3 = call_model()

# Gen LoRA Model and load all adapters
peft_model = set_current_target_domain(model3.clip, lora_type="blora")
oracle_path = os.path.join(output_dir, "result_BLoRA", "ORA")
os.makedirs(oracle_path, exist_ok = True)

for current_target_domain_name in target_domains[:]:
    print(current_target_domain_name)

    t0 = time.time()
    mIoU, weights   = validate_oracle( model3, 
                                        peft_model,
                                        source_domains_dict[current_target_domain_name][2],
                                        lora_type="blora",
                                        )
    # convert numpy.float64 เป็น float ก่อน
    total = time.time() - t0

    print(current_target_domain_name, float(mIoU.avg))
    data_to_save = {
        "Target": current_target_domain_name,
        "mIoU": float(mIoU.avg),
        "Time": total,
        "weights": weights,
        
    }
    savepath = os.path.join(oracle_path, f"{current_target_domain_name}.json")
    with open(savepath, "w") as f:
        json.dump(data_to_save, f, indent=4)

Truly Reload Base Model
Log-In All LoRAs Library
Load First LoRA adapter: ACDC-fog
ACDC-fo LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-fog' loaded
ACDC-ni LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-night' loaded
ACDC-ra LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-rain' loaded


ACDC-sn LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ACDC-snow' loaded
ADE20K LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/ADE20K' loaded
CS LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/CS' loaded
MUSES-c LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-clear' loaded
MUSES-f LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-fog' loaded
MUSES-r LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-rain' loaded


MUSES-s LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MUSES-snow' loaded
BDD LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/BDD' loaded
MV LoRA: 	'/home/chenyinjia/domain-lora/BLORA-SEG/Rebirth/HugFace/output/2base8bz/blora-db/MV' loaded
ACDC-fog


Validation:   0%|          | 0/100 [00:00<?, ?it/s]

Average mIoU: 0.45670761955127726
ACDC-fog 0.45670761955127726
ACDC-night


...

Average mIoU: 0.3811132203507302
MUSES-snow 0.3811132203507302
BDD


Validation:   0%|          | 0/1000 [00:00<?, ?it/s]

Average mIoU: 0.46025761132509596
BDD 0.46025761132509596
MV


Validation:   0%|          | 0/2000 [00:00<?, ?it/s]

Average mIoU: 0.5462172110477282
MV 0.5462172110477282
